# 1. Football Expected Goals Prediction

We estimate the probability that a non-penalty football shot becomes a goal and ask whether shot context improves prediction beyond distance and angle. We compare Logistic Regression and Support Vector Machine (SVM) models using two feature sets: distance and angle alone, and distance and angle with shot context. The predicted probability of a goal is the shot’s expected goals (xG) value.

Run the cells in order in a fresh copy of the repository. In Google Colab, upload the repository ZIP when prompted. The StatsBomb extraction needs an internet connection and can take time. A full run regenerates the data, predictions, metrics, and model files in that copy.


## Setup

Install the libraries used by the extraction, feature engineering, and modelling stages, then locate the repository once for the whole notebook.


In [ ]:
%pip install -q pandas numpy matplotlib mplsoccer scikit-learn scipy statsbombpy tqdm joblib


In [ ]:
from pathlib import Path
import sys
import zipfile

DATA_FILE = "data/processed/modeling_dataset.csv"
project_root = Path.cwd()
if not (project_root / DATA_FILE).exists():
    if (project_root.parent / DATA_FILE).exists():
        project_root = project_root.parent
    else:
        from google.colab import files
        print("Upload the IT461 project repository ZIP:")
        uploaded = files.upload()
        zip_name = next(name for name in uploaded if name.lower().endswith(".zip"))
        with zipfile.ZipFile(zip_name) as archive:
            archive.extractall(Path.cwd())
        project_root = next(
            path for path in Path.cwd().iterdir()
            if path.is_dir() and (path / DATA_FILE).exists()
        )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project folder:", project_root)


# 2. Dataset Preparation and Extraction

We collect non-penalty shots from a reproducible sample of 500 StatsBomb Open Data matches and retain the fields used later in the project.


In [ ]:
import pandas as pd
from statsbombpy import sb
from tqdm.auto import tqdm

TARGET_MATCHES = 500
RANDOM_STATE = 42

RAW_INPUT_FIELDS = [
    "location",
    "shot_body_part",
    "shot_technique",
    "shot_first_time",
    "shot_type",
    "play_pattern",
    "under_pressure",
]

KEEP_COLUMNS = [
    "match_id",
    "id",
    "team",
    "player",
    *RAW_INPUT_FIELDS,
    "shot_outcome",
    "shot_statsbomb_xg",
]


## 2.1 Get available matches

In [ ]:
competitions = sb.competitions()

match_frames = []

competition_seasons = competitions[
    ["competition_id", "season_id"]
].drop_duplicates()

for row in tqdm(
    competition_seasons.itertuples(index=False),
    total=len(competition_seasons),
    desc="Loading match lists"
):
    try:
        matches = sb.matches(
            competition_id=int(row.competition_id),
            season_id=int(row.season_id)
        )

        match_columns = [
            c for c in
            ["match_id", "match_date", "competition", "season", "home_team", "away_team"]
            if c in matches.columns
        ]

        match_frames.append(matches[match_columns])

    except Exception as e:
        print(
            f"Skipped competition {row.competition_id}, "
            f"season {row.season_id}: {e}"
        )

all_matches = (
    pd.concat(match_frames, ignore_index=True)
      .drop_duplicates(subset="match_id")
      .reset_index(drop=True)
)

candidate_matches = (
    all_matches.sample(frac=1, random_state=RANDOM_STATE)
               .reset_index(drop=True)
)

print(f"Available open-data matches: {len(all_matches):,}")
print(f"Target sample: {min(TARGET_MATCHES, len(all_matches)):,} matches")


## 2.2 Extract non-penalty shots

In [ ]:
shot_frames = []
successful_match_ids = []
failed_match_ids = []

for match_id in tqdm(candidate_matches["match_id"], desc="Loading events"):
    if len(successful_match_ids) >= TARGET_MATCHES:
        break

    try:
        events = sb.events(match_id=int(match_id))

        match_shots = events.loc[events["type"].eq("Shot")].copy()
        match_shots = match_shots.loc[
            ~match_shots["shot_type"].eq("Penalty")
        ].copy()

        # Keep the selected raw fields only.
        for column in KEEP_COLUMNS:
            if column not in match_shots.columns:
                match_shots[column] = pd.NA

        shot_frames.append(match_shots[KEEP_COLUMNS])
        successful_match_ids.append(int(match_id))

    except Exception as e:
        failed_match_ids.append(int(match_id))
        print(f"Skipped match {match_id}: {e}")

shots = pd.concat(shot_frames, ignore_index=True)

used_matches = all_matches[
    all_matches["match_id"].isin(successful_match_ids)
].copy()

print(f"Matches loaded: {len(successful_match_ids):,}")
print(f"Non-penalty shots: {len(shots):,}")
print(f"Saved columns: {len(shots.columns)}")
print(f"Failed matches: {len(failed_match_ids)}")

shots.head()


## 2.3 Dataset summary

In [ ]:
goal_count = shots["shot_outcome"].eq("Goal").sum()
non_goal_count = len(shots) - goal_count
goal_rate = goal_count / len(shots) if len(shots) else 0

summary = pd.DataFrame({
    "Statistic": [
        "Matches",
        "Non-penalty shots",
        "Raw model input fields",
        "Total columns in saved shot file",
        "Target classes",
        "Goals",
        "Non-goals",
        "Goal rate",
    ],
    "Value": [
        len(successful_match_ids),
        len(shots),
        len(RAW_INPUT_FIELDS),
        len(shots.columns),
        2,
        int(goal_count),
        int(non_goal_count),
        f"{goal_rate:.2%}",
    ]
})

display(summary)


In [ ]:
feature_summary = pd.DataFrame({
    "Raw field": RAW_INPUT_FIELDS,
    "Role": [
        "Shot location coordinates",
        "Body part used for the shot",
        "Shot technique",
        "Whether the shot was first-time",
        "Shot type",
        "Play pattern leading to the shot",
        "Whether the shooter was under pressure",
    ]
})

display(feature_summary)


## 2.4 Representative examples

In [ ]:
goal_examples = shots[shots["shot_outcome"].eq("Goal")].sample(
    n=min(3, goal_count),
    random_state=RANDOM_STATE
)

non_goal_examples = shots[~shots["shot_outcome"].eq("Goal")].sample(
    n=min(3, non_goal_count),
    random_state=RANDOM_STATE
)

example_rows = pd.concat(
    [goal_examples, non_goal_examples],
    ignore_index=True
)

display(
    example_rows[
        [
            "location",
            "shot_body_part",
            "shot_technique",
            "shot_first_time",
            "shot_type",
            "play_pattern",
            "under_pressure",
            "shot_outcome",
        ]
    ]
)


## 2.5 Export

In [ ]:
raw_dir = project_root / "data" / "raw"
metadata_dir = project_root / "data" / "metadata"

# Create the folders if they do not already exist.
raw_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

shots.to_csv(
    raw_dir / "statsbomb_non_penalty_shots_raw.csv",
    index=False
)

used_matches.to_csv(
    metadata_dir / "statsbomb_selected_matches.csv",
    index=False
)

summary.to_csv(
    metadata_dir / "statsbomb_dataset_summary.csv",
    index=False
)

print("Saved:")
print("- data/raw/statsbomb_non_penalty_shots_raw.csv")
print("- data/metadata/statsbomb_selected_matches.csv")
print("- data/metadata/statsbomb_dataset_summary.csv")

**Note:** `shot_statsbomb_xg` is retained only as a later benchmark. It must not be used as a model input. Distance, angle, encoded categories, and the binary goal target are intentionally not created in this preparation stage.

# 3. Data Cleaning

We handle missing Boolean flags, create the binary goal target, and check the data before modelling.


In [ ]:
import pandas as pd

# Load the raw dataset produced by the data-preparation stage.
raw_path = project_root / "data" / "raw" / "statsbomb_non_penalty_shots_raw.csv"
df = pd.read_csv(raw_path)

# Preserve a copy of the original dataset.
df_raw = df.copy()

print("Dataset shape:", df.shape)
display(df.head())
df.info()

## 3.1 Handling Missing Values

Missing values were identified in the **shot_first_time** and **under_pressure** attributes. Since these attributes represent Boolean flags, missing values are treated as False, indicating that the corresponding condition was not recorded as present. This approach preserves all shot records without removing potentially useful training data.

In [ ]:
# Check missing values in each column
print("Missing values before cleaning:")
print(df.isnull().sum())


In [ ]:
# Replace missing values with False
df["shot_first_time"] = df["shot_first_time"].fillna(False)
df["under_pressure"] = df["under_pressure"].fillna(False)


print("Missing values after cleaning:")
print(df.isnull().sum())

# Check the data types
print(df[["shot_first_time", "under_pressure"]].dtypes)


## 3.2 Creating the Binary Target Variable

The **shot_outcome** column contains different shot outcomes. Since this project is a binary classification task, a new target variable named goal is created, where 1 represents a goal and 0 represents any non-goal outcome.

In [ ]:

# Check the original shot outcomes
print("Original shot outcomes:")
print(df["shot_outcome"].value_counts())

# binary target variable
df["goal"] = (df["shot_outcome"] == "Goal").astype(int)

# the new target variable
print("\nBinary target distribution:")
print(df["goal"].value_counts())

# Display sample records
display(df[["shot_outcome", "goal"]].head())


## 3.3 Data Quality Verification

After handling missing values and creating the binary target variable, the dataset is checked for data types, duplicate records, and target class distribution. These checks ensure data consistency and help identify potential issues before model training.

In [ ]:
# Check data types
print("Data types:")
print(df.dtypes)

# Check duplicate rows and shot IDs
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate shot IDs:", df["id"].duplicated().sum())

# Check class distribution
print("\nClass distribution (counts):")
print(df["goal"].value_counts())

print("\nClass distribution (percentages):")
print((df["goal"].value_counts(normalize=True) * 100).round(2))

# Check final dataset shape
print("\nDataset shape:", df.shape)


## 3.4 Reusable Data-Cleaning Function

A reusable function is created to combine the preprocessing steps into a single process. The function handles missing Boolean values and creates the binary target variable while preserving the original dataset. This ensures that the same cleaning steps can be applied consistently whenever needed.

In [ ]:
def clean_shot_data(raw_df):
    # Create a copy to preserve the original dataset
    cleaned_df = raw_df.copy()

    # Handle missing values
    cleaned_df["shot_first_time"] = cleaned_df["shot_first_time"].fillna(False)
    cleaned_df["under_pressure"] = cleaned_df["under_pressure"].fillna(False)

    # Create the binary target variable
    cleaned_df["goal"] = (cleaned_df["shot_outcome"] == "Goal").astype(int)

    return cleaned_df


# Apply the cleaning function to the original dataset
df_clean = clean_shot_data(df_raw)

print("Cleaned dataset shape:", df_clean.shape)
print("Total missing values:", df_clean.isnull().sum().sum())
print("Duplicate shot IDs:", df_clean["id"].duplicated().sum())

display(df_clean.head())


In [ ]:
from pathlib import Path

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = processed_dir / "cleaned_shots.csv"
df_clean.to_csv(cleaned_path, index=False)

print("Saved cleaned dataset:", cleaned_path)
print("Shape:", df_clean.shape)

# 4. Feature Engineering, Preprocessing, and Splitting

We derive shot distance and angle, define two feature sets, and assign entire matches to training or testing.


## Input from the Data Cleaning Stage

The data-cleaning stage handles missing values, creates the binary goal target, and exports cleaned_shots.csv.

This stage **does not repeat those cleaning steps**. It loads that exported dataset and continues from there.

In [ ]:
import pandas as pd

# Add the project root so shared project modules can be imported reliably.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Define the shared processed-data directory once for the whole notebook.
processed_dir = project_root / "data" / "processed"

# Load the cleaned dataset produced by the previous data-cleaning stage.
df_clean = pd.read_csv(processed_dir / "cleaned_shots.csv")

# Verify that the cleaned input is complete and consistent.
print("Cleaned dataset shape:", df_clean.shape)
print("Total missing values:", df_clean.isna().sum().sum())
print("Duplicate shot IDs:", df_clean["id"].duplicated().sum())

# Inspect the binary target distribution.
print("\nTarget distribution:")
print(df_clean["goal"].value_counts())
print((df_clean["goal"].value_counts(normalize=True) * 100).round(2))

# Display a small sample of the cleaned data.
display(df_clean.head())


## 4.1. Feature Engineering

### 4.1.1 Shot Location Processing

The raw `location` attribute stores the position of each shot as an `[x, y]` coordinate pair. The values are converted into a numerical format and separated into individual `x` and `y` components.

These coordinates are then used to calculate the shot's **distance from goal** and **opening angle to the goalposts**.

In [ ]:
import ast

# Create a separate dataframe for feature engineering
df_features = df_clean.copy()

# Convert the stored location string into a Python list
df_features["location"] = df_features["location"].apply(ast.literal_eval)

# Separate each location into numeric x and y coordinates
df_features["x"] = df_features["location"].apply(lambda loc: loc[0])
df_features["y"] = df_features["location"].apply(lambda loc: loc[1])

# Inspect the parsed coordinates
display(df_features[["location", "x", "y"]].head())

### Understanding the StatsBomb Coordinate System

StatsBomb represents the pitch using a **120 × 80 coordinate system**. `x` describes the position along the pitch length and `y` describes the position across its width. Shots are represented toward the attacking goal, whose centre is at **(120, 40)**.

The figure below illustrates the coordinate system and an example shot at **(96.5, 57.7)**.

In [ ]:
from mplsoccer import Pitch
import matplotlib.pyplot as plt

# Example shot taken from the dataset
example_x = 96.5
example_y = 57.7

# Create a StatsBomb pitch using the 120 x 80 coordinate system
pitch = Pitch(
    pitch_type="statsbomb",
    pitch_color="#2f6b45",
    line_color="#f2f2f2",
    linewidth=1.5,
    stripe=True,
    stripe_color="#326f49"
)

fig, ax = pitch.draw(figsize=(8, 6))

# Label the four pitch corners to make the coordinate system clear
corner_labels = [
    (0, 0, "(0, 0)", (3, 3)),
    (120, 0, "(120, 0)", (103, 3)),
    (0, 80, "(0, 80)", (3, 76)),
    (120, 80, "(120, 80)", (100, 76))
]

for x, y, label, text_pos in corner_labels:
    pitch.annotate(
        label, (x, y), ax=ax, xytext=text_pos,
        fontsize=9, color="white", fontweight="bold"
    )

# Mark the two goalposts at approximately y = 36 and y = 44
pitch.scatter(
    [120, 120], [36, 44], ax=ax, s=65,
    color="white", edgecolor="black", zorder=5
)

# Mark and label the centre of the attacking goal
pitch.scatter(
    120, 40, ax=ax, s=110,
    color="#e63946", edgecolor="white", linewidth=1.5, zorder=6
)

pitch.annotate(
    "Goal centre\n(120, 40)", (120, 40), ax=ax, xytext=(102, 31),
    fontsize=10, color="white", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="white", lw=1.2)
)

# Plot and label the example shot
pitch.scatter(
    example_x, example_y, ax=ax, s=160,
    color="#ffd166", edgecolor="black", linewidth=1.2, zorder=7
)

pitch.annotate(
    "Example shot\n(96.5, 57.7)", (example_x, example_y), ax=ax,
    xytext=(72, 67), fontsize=10, color="white", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="white", lw=1.2)
)

# Draw a dashed line from the example shot to the goal centre
pitch.lines(
    example_x, example_y, 120, 40, ax=ax,
    color="#ffd166", linestyle="--", linewidth=1.5, alpha=0.9, zorder=4
)

# Show the attacking direction
pitch.annotate(
    "Attacking direction", (78, 7), ax=ax, xytext=(38, 7),
    fontsize=10, color="white", fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="white", lw=1.5)
)

# Add the figure title
ax.set_title(
    "StatsBomb Pitch Coordinate System",
    fontsize=16, fontweight="bold", pad=14
)

plt.show()

### 4.1.2 Distance to Goal

Shot distance measures how far the shot location is from the centre of the attacking goal at **(120, 40)**.

Using the shot coordinates `(x, y)`, the Euclidean distance is:

$$
\text{distance} = \sqrt{(120-x)^2 + (40-y)^2}
$$

The value is expressed in **StatsBomb coordinate units**, not metres.

In [ ]:
import numpy as np

# Calculate Euclidean distance from each shot to the goal centre at (120, 40)
df_features["distance"] = np.sqrt(
    (120 - df_features["x"])**2 +
    (40 - df_features["y"])**2
)

# Inspect the engineered distance feature
display(df_features[["location", "x", "y", "distance"]].head())

### 4.1.3 Shot Angle

Shot angle describes **how wide the goal appears from the shot location**. It is the opening angle between the two goalposts at approximately `(120, 36)` and `(120, 44)`.

The angle is stored in **radians** for modelling. A degree version is calculated only for easier inspection.

In [ ]:
# Calculate the direction from each shot to the two goalposts
angle_post_1 = np.arctan2(
    36 - df_features["y"],
    120 - df_features["x"]
)

angle_post_2 = np.arctan2(
    44 - df_features["y"],
    120 - df_features["x"]
)

# Calculate the opening angle between the two goalpost directions
df_features["angle"] = np.abs(angle_post_2 - angle_post_1)

# Use the smaller angle if wrap-around occurs across the -pi / pi boundary
df_features["angle"] = np.minimum(
    df_features["angle"],
    2 * np.pi - df_features["angle"]
)

# Create a degree version only for easier human inspection
df_features["angle_degrees"] = np.degrees(df_features["angle"])

# Inspect the engineered angle feature
display(
    df_features[
        ["location", "distance", "angle", "angle_degrees"]
    ].head()
)

### 4.1.4 Engineered Feature Validation

The engineered features are checked for missing and infinite values and inspected to ensure that distance is non-negative and angle remains within its valid range.

In [ ]:
# Check the engineered numerical features for missing values
print("Missing distance values:", df_features["distance"].isna().sum())
print("Missing angle values:", df_features["angle"].isna().sum())

# Check that no infinite values were produced
print("Infinite distance values:", np.isinf(df_features["distance"]).sum())
print("Infinite angle values:", np.isinf(df_features["angle"]).sum())

# Inspect the numerical ranges of the engineered features
print("\nDistance range:")
print(df_features["distance"].min(), "to", df_features["distance"].max())

print("\nAngle range (radians):")
print(df_features["angle"].min(), "to", df_features["angle"].max())

print("\nAngle range (degrees):")
print(df_features["angle_degrees"].min(), "to", df_features["angle_degrees"].max())

## 4.2. Shared Modelling Dataset

### 4.2.1 Feature Set Definition

The project uses two feature configurations: a location-only baseline and a context-aware version. To keep the feature definitions identical across feature engineering, Logistic Regression, and SVM, the shared feature lists are imported from `src/preprocessing.py` rather than being defined separately in multiple places.


In [ ]:
# Import the shared feature definitions and preprocessing functions.
# The project root was added to sys.path when the cleaned data was loaded.
from src.preprocessing import (
    LOCATION_FEATURES,
    CONTEXT_FEATURES,
    TARGET,
    BENCHMARK,
    make_location_preprocessor,
    make_context_preprocessor,
)

# Use the shared feature definitions.
location_features = LOCATION_FEATURES
context_features = CONTEXT_FEATURES
target = TARGET

# Create the two feature matrices and the shared target vector.
X_location = df_features[location_features].copy()
X_context = df_features[context_features].copy()
y = df_features[target].copy()

# Inspect the resulting dataset dimensions.
print("Location-only features:", location_features)
print("Context-aware features:", context_features)
print("Target:", target)
print("Benchmark (not used for training):", BENCHMARK)
print("\nLocation-only shape:", X_location.shape)
print("Context-aware shape:", X_context.shape)
print("Target shape:", y.shape)


### 4.2.2 Exporting the Shared Modelling Dataset

The shared modelling dataset is exported **before scaling and one-hot encoding**.

This lets Logistic Regression and SVM start from exactly the same engineered data while fitting preprocessing only on their training data. It also keeps preprocessing safe for cross-validation.

In [ ]:
# Ensure the shared processed-data directory exists.
processed_dir.mkdir(parents=True, exist_ok=True)

# Select only the columns needed by the modelling stage
# shot_statsbomb_xg is included for later benchmarking, not as a training feature
modeling_columns = [
    "id",
    "match_id",
    "distance",
    "angle",
    "shot_body_part",
    "shot_technique",
    "shot_first_time",
    "shot_type",
    "play_pattern",
    "under_pressure",
    "goal",
    "shot_statsbomb_xg"
]

# Create the shared, unscaled and unencoded modelling dataset
modeling_df = df_features[modeling_columns].copy()

# Export the dataset so all model notebooks start from the same engineered data
modeling_path = processed_dir / "modeling_dataset.csv"
modeling_df.to_csv(modeling_path, index=False)

print("Saved:", modeling_path)
print("Shape:", modeling_df.shape)
display(modeling_df.head())

## 4.3. Shared Match-Level Train-Test Split

The shared train-test assignment is created **by `match_id` rather than by individual shot**. All shots from the same match are kept entirely in either training or testing.

`StratifiedGroupKFold` is used with five folds, and the first fold is used as the fixed test set. This gives an approximately 80/20 split, preserves the goal distribution closely, and prevents shots from the same match appearing in both sets.

Because the split is performed using **whole matches**, and different matches contain different numbers of shots, the final number of shots in each set will be approximately rather than exactly 80/20.

The assignments are saved to `model_split.csv` so both model sections use the exact same split.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Create a reproducible grouped split.
# Grouping by match_id keeps every match entirely in train or entirely in test.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Use the first fold as the fixed test set.
# Since whole matches are assigned together, the shot counts are approximately,
# rather than exactly, 80/20.
train_pos, test_pos = next(
    splitter.split(
        df_features,
        y=y,
        groups=df_features["match_id"]
    )
)

# Convert the positional indices returned by scikit-learn to dataframe index labels
train_idx = df_features.index[train_pos]
test_idx = df_features.index[test_pos]

# Save a stable split assignment using the unique shot ID and its match ID
split_df = df_features[["id", "match_id"]].copy()
split_df["split"] = "train"
split_df.loc[test_idx, "split"] = "test"

split_path = processed_dir / "model_split.csv"
split_df.to_csv(split_path, index=False)

# Create train/test objects only to demonstrate and validate preprocessing below
X_location_train = X_location.loc[train_idx].copy()
X_location_test = X_location.loc[test_idx].copy()

X_context_train = X_context.loc[train_idx].copy()
X_context_test = X_context.loc[test_idx].copy()

y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

# Keep StatsBomb xG aligned with the fixed test set for later benchmarking
statsbomb_xg_test = df_features.loc[test_idx, "shot_statsbomb_xg"].copy()

print("Saved:", split_path)

### 4.3.1 Split Validation

The split is checked to confirm that no `match_id` appears in both the training and testing sets. The target distribution is also reported for both subsets to verify that the proportion of goals and non-goals remains similar after grouping by match.

In [ ]:
# Collect the match IDs assigned to each subset
train_matches = set(df_features.loc[train_idx, "match_id"])
test_matches = set(df_features.loc[test_idx, "match_id"])

# Verify that no match appears in both training and testing
match_overlap = train_matches.intersection(test_matches)

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))
print("Training matches:", len(train_matches))
print("Testing matches:", len(test_matches))
print("Matches appearing in both sets:", len(match_overlap))

# Report the target distribution in the training set
print("\nTraining target distribution:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

# Report the target distribution in the testing set
print("\nTesting target distribution:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))

## 4.4. Shared Preprocessing Strategy

### 4.4.1 Encoding and Scaling

The models use the same preprocessing rules:

- `distance` and `angle` are standardized using `StandardScaler`.
- categorical variables are one-hot encoded with `OneHotEncoder(handle_unknown="ignore")`.
- Boolean variables are converted to `0/1`.

These definitions live in `src/preprocessing.py`, which acts as the **single shared source** for both models and this preprocessing demonstration. The transformations are fitted using training data only so information from the test set is not used when learning scaling or encoding parameters.


In [ ]:
# Create fresh, unfitted preprocessors from the shared module.
# The transformation logic is defined only in src/preprocessing.py so
# Logistic Regression and SVM cannot accidentally use different versions.
location_preprocessor = make_location_preprocessor()
context_preprocessor = make_context_preprocessor()

print("Shared preprocessing pipelines created successfully.")


### 4.4.2 Preprocessing Demonstration and Validation

The shared preprocessors are fitted below only to verify that they work correctly on the fixed train-test split. This demonstration does **not** create transformed files for model training.

In the Logistic Regression and SVM sections, a fresh preprocessor is created inside each sklearn `Pipeline`. During hyperparameter tuning, this allows scaling and encoding to be refitted within each cross-validation training fold and prevents leakage from the corresponding validation fold.


In [ ]:
# Fit the shared preprocessors using training data only, then apply the
# learned transformations unchanged to the held-out test data.

# Location-only preprocessing
X_location_train_processed = location_preprocessor.fit_transform(
    X_location_train
)
X_location_test_processed = location_preprocessor.transform(
    X_location_test
)

# Context-aware preprocessing
X_context_train_processed = context_preprocessor.fit_transform(
    X_context_train
)
X_context_test_processed = context_preprocessor.transform(
    X_context_test
)

# Inspect the resulting matrix dimensions.
print("Location-only:")
print("Training shape:", X_location_train_processed.shape)
print("Testing shape:", X_location_test_processed.shape)

print("\nContext-aware:")
print("Training shape:", X_context_train_processed.shape)
print("Testing shape:", X_context_test_processed.shape)


In [ ]:
from scipy import sparse

# Helper function that works with both dense and sparse transformed matrices.
def validate_processed_data(name, X):
    values = X.data if sparse.issparse(X) else np.asarray(X)

    print(name)
    print("Shape:", X.shape)
    print("Missing values:", np.isnan(values).sum())
    print("Infinite values:", np.isinf(values).sum())
    print()

# Validate all four transformed datasets.
validate_processed_data(
    "Location-only training data",
    X_location_train_processed
)
validate_processed_data(
    "Location-only test data",
    X_location_test_processed
)
validate_processed_data(
    "Context-aware training data",
    X_context_train_processed
)
validate_processed_data(
    "Context-aware test data",
    X_context_test_processed
)

# Retrieve the feature names created by the fitted shared preprocessors.
location_feature_names = location_preprocessor.get_feature_names_out()
context_feature_names = context_preprocessor.get_feature_names_out()

# Confirm that StandardScaler was fitted correctly on training data.
print("Scaled location-only training feature means:")
print(
    dict(
        zip(
            location_feature_names,
            np.round(X_location_train_processed.mean(axis=0), 4)
        )
    )
)

print("\nScaled location-only training feature standard deviations:")
print(
    dict(
        zip(
            location_feature_names,
            np.round(X_location_train_processed.std(axis=0), 4)
        )
    )
)

# Show exactly how the original context-aware columns expand after preprocessing.
print("\nNumber of processed context-aware features:", len(context_feature_names))
print("Processed context-aware feature names:")
for feature_name in context_feature_names:
    print("-", feature_name)


## 4.5. Shared Modelling Inputs

The shared resources used by the modelling stage are:

- `data/processed/modeling_dataset.csv`
- `data/processed/model_split.csv`
- `src/preprocessing.py`

The Logistic Regression and SVM sections load the same modelling dataset and split file, then create a fresh preprocessor from `src/preprocessing.py` inside each model pipeline. This keeps the feature definitions and transformations identical across both models and allows preprocessing to be refitted safely during cross-validation.

`shot_statsbomb_xg` remains benchmark-only and must not be included as a training feature.


# 5. Logistic Regression

Logistic Regression gives a direct probability estimate for each shot. We tune location-only and context-aware models on match-separated training folds, then evaluate them on held-out matches.


## 5.1 Libraries

The repository was located in the common setup. We now load the modelling libraries and shared preprocessing functions.

In [ ]:
import time
import platform
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_val_predict
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, log_loss, brier_score_loss,
    average_precision_score, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, classification_report,
)

from src.preprocessing import (
    LOCATION_FEATURES, CONTEXT_FEATURES,
    make_location_preprocessor, make_context_preprocessor,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*penalty is deprecated.*")  # warning only shows in newer sklearn
print("Python:", platform.python_version(), "| scikit-learn:", sklearn.__version__)
print("Libraries and shared preprocessing loaded.")

## 5.2 Loading the data

`modeling_dataset.csv` has the engineered features but they are not scaled or encoded yet. That is on purpose: the scaler and encoder have to learn from the training data only, otherwise information from the test set leaks into the model.

`model_split.csv` says which shots are in train and which are in test. We join the two files on the shot ID and match ID (not on row order) and then check that no match ends up in both sets.

In [ ]:
data = pd.read_csv(project_root / "data/processed/modeling_dataset.csv")
splits = pd.read_csv(project_root / "data/processed/model_split.csv")

# Join on the IDs so we never rely on the row order of the two files.
assert data["id"].is_unique and splits["id"].is_unique
shots = data.merge(splits, on=["id", "match_id"], validate="one_to_one")
assert len(shots) == len(data) == len(splits)
assert set(shots["split"]) == {"train", "test"}
assert set(shots["goal"].unique()) == {0, 1}
assert shots[CONTEXT_FEATURES].notna().all().all()

train = shots[shots["split"] == "train"].reset_index(drop=True)
test = shots[shots["split"] == "test"].reset_index(drop=True)

# A match should be either in train or in test, never both.
assert set(train["match_id"]).isdisjoint(set(test["match_id"]))
print("Total shots:", len(shots), "| Total matches:", shots["match_id"].nunique())
print("Training:", len(train), "shots |", train["match_id"].nunique(), "matches")
print("Testing: ", len(test), "shots |", test["match_id"].nunique(), "matches")
print("Match overlap: 0")

In [ ]:
# Goal rate in train and test should be about the same (~11%).
class_counts = shots.groupby("split")["goal"].agg(
    shots="count", goals="sum", goal_rate="mean"
)
class_counts["non_goals"] = class_counts["shots"] - class_counts["goals"]
display(class_counts[["shots", "non_goals", "goals", "goal_rate"]].round(4))

## 5.3 Feature sets and CV folds

Logistic Regression multiplies each feature by a weight, adds them up, and passes the result through the sigmoid function so the output is always between 0 and 1:

$$P(\text{goal}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots + \beta_k x_k)}}$$

Scaling the numeric features matters here because regularization penalizes all coefficients in the same way, so the features need to be on a similar scale. The preprocessing sits inside the pipeline, which means it gets refitted on every training fold.

For cross-validation We use 3-fold `StratifiedGroupKFold` grouped by `match_id`, with the same random seed as the SVM section. This way both models are tuned on exactly the same folds, and shots from the same match never end up split between training and validation.

In [ ]:
y_train = train["goal"]
y_test = test["goal"]

X_train_location = train[LOCATION_FEATURES]
X_test_location = test[LOCATION_FEATURES]
X_train_context = train[CONTEXT_FEATURES]
X_test_context = test[CONTEXT_FEATURES]

# Same splitter and seed as the SVM notebook, so both use identical folds.
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
cv_folds = list(cv.split(X_train_location, y_train, groups=train["match_id"]))

for number, (train_idx, val_idx) in enumerate(cv_folds, 1):
    train_matches = set(train.loc[train_idx, "match_id"])
    val_matches = set(train.loc[val_idx, "match_id"])
    assert train_matches.isdisjoint(val_matches)
    print(f"Fold {number}: {len(train_idx)} training shots, {len(val_idx)} validation shots")

## 5.4 Tuning the model

We tune three settings:

- `C` controls how strong the regularization is (it is the inverse, so smaller C means stronger). A very small C pushes the coefficients towards zero and the model becomes too simple. A very large C lets them grow freely, which can overfit, especially on categories with only a few shots. We try values from 0.001 to 100.
- `penalty` is the type of regularization. L2 shrinks all coefficients a bit, while L1 can set some of them to exactly zero, so it also works as a kind of feature selection. We use the `liblinear` solver because it supports both.
- `class_weight` is either None or balanced. Only about 11% of shots are goals, so like the SVM section We check whether giving goals more weight helps.

That gives 6 x 2 x 2 = 24 combinations for each feature set.

To pick the best combination We use the mean validation log loss. The output of this project is a probability, and log loss checks whether the probabilities themselves are right, not only whether goals get higher scores than non-goals. We still record ROC AUC so the results can be compared with the SVM. The test set is not touched during tuning.

In [ ]:
parameter_grid = {
    "logreg__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "logreg__penalty": ["l2", "l1"],
    "logreg__class_weight": [None, "balanced"],
}
# sklearn maximizes scores, so log loss is used as a negative number
scoring = {"log_loss": "neg_log_loss", "roc_auc": "roc_auc"}

print("Grid candidates per feature set:",
      len(parameter_grid["logreg__C"]) * len(parameter_grid["logreg__penalty"])
      * len(parameter_grid["logreg__class_weight"]),
      "(each tested across 3 folds)")

In [ ]:
def cv_table(search):
    """Turn GridSearchCV results into a smaller table, best log loss first."""
    results = pd.DataFrame(search.cv_results_)
    table = pd.DataFrame({
        "C": results["param_logreg__C"],
        "Penalty": results["param_logreg__penalty"],
        "Class weight": results["param_logreg__class_weight"].fillna("None"),
        "Mean CV log loss": -results["mean_test_log_loss"],
        "Log loss std. dev.": results["std_test_log_loss"],
        "Mean CV ROC AUC": results["mean_test_roc_auc"],
        "AUC std. dev.": results["std_test_roc_auc"],
    })
    return table.sort_values("Mean CV log loss").reset_index(drop=True)

### 5.4.1 Location-only model

Grid search over the 24 combinations using only `distance` and `angle`.

In [ ]:
location_pipeline = Pipeline([
    ("preprocessing", make_location_preprocessor()),
    ("logreg", LogisticRegression(solver="liblinear", max_iter=5000)),
])

location_search = GridSearchCV(
    location_pipeline, parameter_grid, scoring=scoring,
    refit="log_loss", cv=cv_folds, n_jobs=1  # refit the best setting on all training data
)
start = time.perf_counter()
location_search.fit(X_train_location, y_train)
location_seconds = time.perf_counter() - start

print("Location-only best parameters:", location_search.best_params_)
print("Location-only best CV log loss:", round(-location_search.best_score_, 4))
print(f"Grid search time: {location_seconds:.1f} seconds")

In [ ]:
display(cv_table(location_search).round(4))

### 5.4.2 Context-aware model

Same grid and same folds, this time with all eight features.

In [ ]:
context_pipeline = Pipeline([
    ("preprocessing", make_context_preprocessor()),
    ("logreg", LogisticRegression(solver="liblinear", max_iter=5000)),
])

context_search = GridSearchCV(
    context_pipeline, parameter_grid, scoring=scoring,
    refit="log_loss", cv=cv_folds, n_jobs=1
)
start = time.perf_counter()
context_search.fit(X_train_context, y_train)
context_seconds = time.perf_counter() - start

print("Context-aware best parameters:", context_search.best_params_)
print("Context-aware best CV log loss:", round(-context_search.best_score_, 4))
print(f"Grid search time: {context_seconds:.1f} seconds")

In [ ]:
display(cv_table(context_search).round(4))

### 5.4.3 How C affects the results

Validation scores for each value of C (class_weight = None). A very small C hurts both models. After C = 1 the curves are almost flat, so making the model less regularized does not help much beyond that point.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, search in [("Location-only", location_search), ("Context-aware", context_search)]:
    table = cv_table(search)
    for penalty, style in [("l2", "-o"), ("l1", "--s")]:
        sub = table[(table["Class weight"] == "None") & (table["Penalty"] == penalty)].sort_values("C")
        label = f"{name}, {penalty.upper()}"
        axes[0].plot(sub["C"], sub["Mean CV log loss"], style, label=label)
        axes[1].plot(sub["C"], sub["Mean CV ROC AUC"], style, label=label)

for ax, ylabel in zip(axes, ["Mean CV log loss (lower is better)",
                             "Mean CV ROC AUC (higher is better)"]):
    ax.set_xscale("log")
    ax.set_xlabel("C (log scale)")
    ax.set_ylabel(ylabel)
    ax.legend()
fig.suptitle("Logistic Regression: effect of C (training CV)")
plt.tight_layout()
plt.show()

### 5.4.4 Balanced vs regular class weights

With balanced weights the ROC AUC stays almost the same, so the model still ranks the shots in the same order. But the log loss roughly doubles. Balanced weights make the model behave as if goals and misses were equally common, so every predicted probability goes up and no longer matches how often shots actually go in.

This is the opposite of what happened in the SVM section, where balancing was needed to get a reasonable model at all. Since Logistic Regression already gives probabilities, the regular weights are the better choice for xG.

In [ ]:
weight_rows = []
for name, search in [("Location-only", location_search), ("Context-aware", context_search)]:
    table = cv_table(search)
    for weight in ["None", "balanced"]:
        selected = table[table["Class weight"] == weight]
        weight_rows.append({
            "Feature set": name,
            "Class weight": weight,
            "Best mean CV log loss": selected["Mean CV log loss"].min(),
            "Best mean CV ROC AUC": selected["Mean CV ROC AUC"].max(),
        })

display(pd.DataFrame(weight_rows).round(4))

### 5.4.5 L1 vs L2

Best L1 and best L2 result for each feature set (class_weight = None). We also refit the best L1 context model to see which features it drops. If L1 and L2 score about the same and L1 only removes a few weak features, it means the feature set is reasonable and nothing important needs to be removed.

In [ ]:
penalty_rows = []
for name, search in [("Location-only", location_search), ("Context-aware", context_search)]:
    table = cv_table(search)
    table = table[table["Class weight"] == "None"]
    for penalty in ["l2", "l1"]:
        sub = table[table["Penalty"] == penalty]
        best = sub.loc[sub["Mean CV log loss"].idxmin()]
        penalty_rows.append({
            "Feature set": name, "Penalty": penalty.upper(), "Best C": best["C"],
            "Best mean CV log loss": best["Mean CV log loss"],
            "Mean CV ROC AUC": best["Mean CV ROC AUC"],
        })
penalty_table = pd.DataFrame(penalty_rows)
display(penalty_table.round(4))

# Refit the best L1 context model and check which coefficients became zero.
best_l1_C = penalty_table.query("`Feature set` == 'Context-aware' and Penalty == 'L1'")["Best C"].iloc[0]
l1_model = Pipeline([
    ("preprocessing", make_context_preprocessor()),
    ("logreg", LogisticRegression(solver="liblinear", penalty="l1", C=best_l1_C, max_iter=5000)),
]).fit(X_train_context, y_train)
names = [n.split("__", 1)[1] for n in l1_model.named_steps["preprocessing"].get_feature_names_out()]
l1_coefs = pd.Series(l1_model.named_steps["logreg"].coef_[0], index=names)
print(f"L1 (C={best_l1_C}) kept {(l1_coefs != 0).sum()} of {len(l1_coefs)} features.")
print("Features set to zero:", list(l1_coefs[l1_coefs == 0].index) or "none")

## 5.5 Training the final models

The grid search only told us which settings work best. Here we take those settings and train each final model on the whole training set (all 9,900 shots), so every shot in the training data is used.

In [ ]:
def build_model(make_preprocessor, best_params):
    """Logistic Regression pipeline with the settings chosen in tuning."""
    return Pipeline([
        ("preprocessing", make_preprocessor()),
        ("logreg", LogisticRegression(
            solver="liblinear",
            C=best_params["logreg__C"],
            penalty=best_params["logreg__penalty"],
            class_weight=best_params["logreg__class_weight"],
            max_iter=5000,
        )),
    ])

location_model = build_model(make_location_preprocessor, location_search.best_params_)
context_model = build_model(make_context_preprocessor, context_search.best_params_)

# Train both models on the full training set.
location_model.fit(X_train_location, y_train)
context_model.fit(X_train_context, y_train)

for name, model in [("Location-only", location_model), ("Context-aware", context_model)]:
    logreg = model.named_steps["logreg"]
    print(f"{name}: C={logreg.C}, penalty={logreg.penalty}, "
          f"class_weight={logreg.class_weight}, "
          f"trained on {len(y_train)} shots, {logreg.coef_.shape[1]} input features")

In [ ]:
# Quick check: these should match the models GridSearchCV refitted internally.
assert np.allclose(location_model.predict_proba(X_train_location),
                   location_search.best_estimator_.predict_proba(X_train_location))
assert np.allclose(context_model.predict_proba(X_train_context),
                   context_search.best_estimator_.predict_proba(X_train_context))
print("Final models match the best models from the grid search.")

## 5.6 Predicting xG on the test set

Now the test set is used for the first time. Unlike the SVM, Logistic Regression gives probabilities directly, so no calibration step is needed. The second column of `predict_proba` is the probability of a goal, which is the xG.

In [ ]:
# location_model and context_model were trained in section 3.5

# column 1 = probability of a goal
location_xg = location_model.predict_proba(X_test_location)[:, 1]
context_xg = context_model.predict_proba(X_test_context)[:, 1]

example_predictions = test[["id", "goal"]].head(5).copy()
example_predictions["Location xG"] = location_xg[:5]
example_predictions["Context xG"] = context_xg[:5]
display(example_predictions.round(4))

### 5.6.1 Sanity check on the predictions

A useful xG model should give clearly different values to easy and hard chances, not just predict something close to 11% for every shot. The average predicted xG should also be close to the real goal rate.

In [ ]:
probability_summary = pd.DataFrame({
    "Feature set": ["Location-only", "Context-aware"],
    "Mean xG": [location_xg.mean(), context_xg.mean()],
    "Std. dev.": [location_xg.std(), context_xg.std()],
    "Min xG": [location_xg.min(), context_xg.min()],
    "Median xG": [np.median(location_xg), np.median(context_xg)],
    "Max xG": [location_xg.max(), context_xg.max()],
})
display(probability_summary.round(4))
print(f"Actual test goal rate: {y_test.mean():.4f}")

## 5.7 What the model learned

One advantage of Logistic Regression is that we can read the coefficients:

- A positive coefficient means the feature makes a goal more likely, a negative one makes it less likely.
- The odds ratio is e^coefficient. An odds ratio of 2 means the odds of scoring double, and 0.5 means they are cut in half.

Distance and angle are standardized, so their coefficients are per one standard deviation. For the one-hot categories, each coefficient should be compared with the other categories in the same group.

In [ ]:
def coefficient_table(model):
    names = model.named_steps["preprocessing"].get_feature_names_out()
    names = [name.split("__", 1)[1] for name in names]  # drop the "num__"/"cat__" prefix
    coefficients = model.named_steps["logreg"].coef_[0]
    table = pd.DataFrame({
        "Feature": names,
        "Coefficient": coefficients,
        "Odds ratio": np.exp(coefficients),
    })
    return table.sort_values("Coefficient", key=abs, ascending=False).reset_index(drop=True)

print("Location-only model")
display(coefficient_table(location_model).round(3))
print("Intercept:", round(location_model.named_steps["logreg"].intercept_[0], 3))

In [ ]:
context_coefficients = coefficient_table(context_model)
display(context_coefficients.round(3))

plot_data = context_coefficients.sort_values("Coefficient")
colors = ["tab:red" if value < 0 else "tab:green" for value in plot_data["Coefficient"]]
plt.figure(figsize=(8, 7))
plt.barh(plot_data["Feature"], plot_data["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient (change in log-odds of scoring)")
plt.title("Context-aware Logistic Regression coefficients")
plt.tight_layout()
plt.show()

### 5.7.1 A note on rare categories

Some categories appear in only a few shots in the training data. Their coefficients are based on very little evidence and can look large just by chance, so they should not be read as real effects. For example, `shot_type_Corner` has one of the biggest coefficients but only 3 training shots.

In [ ]:
category_counts = []
for feature in ["shot_body_part", "shot_technique", "shot_type", "play_pattern"]:
    counts = train.groupby(feature)["goal"].agg(shots="count", goals="sum")
    for category, row in counts.iterrows():
        category_counts.append({
            "Feature": f"{feature}_{category}",
            "Training shots": row["shots"],
            "Training goals": row["goals"],
        })

category_counts = pd.DataFrame(category_counts)
rare = category_counts.merge(context_coefficients, on="Feature")
rare = rare[rare["Training shots"] < 100].sort_values("Training shots")
print("Categories with fewer than 100 training shots:")
display(rare.round(3))

## 5.8 Evaluation on the test set

Almost 89% of shots are not goals, so a model that always says "no goal" already gets 89% accuracy. That is why accuracy is not enough here. We report:

- Precision, recall and F1 for the yes/no predictions
- ROC AUC and average precision for how well goals are ranked above non-goals
- Log loss and Brier score for how accurate the probabilities are

The baseline gives every shot the same xG (the training goal rate). To match the SVM section, the yes/no metrics use a threshold of 0.5. Recall at 0.5 is low because the model is rarely more than 50% sure about a goal, which is normal for shots. Section 5.8.6 looks at a better threshold.

In [ ]:
# Baseline: same xG for every shot, equal to the training goal rate.
training_goal_rate = y_train.mean()
baseline_xg = np.full(len(test), training_goal_rate)

def evaluate_probabilities(name, probabilities, threshold=0.5):
    labels = (probabilities >= threshold).astype(int)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, labels),
        "Precision": precision_score(y_test, labels, zero_division=0),
        "Recall": recall_score(y_test, labels, zero_division=0),
        "F1": f1_score(y_test, labels, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, probabilities),
        "Average precision": average_precision_score(y_test, probabilities),
        "Log loss": log_loss(y_test, probabilities),
        "Brier score": brier_score_loss(y_test, probabilities),
    }

metrics = pd.DataFrame([
    evaluate_probabilities("Training-rate baseline", baseline_xg),
    evaluate_probabilities("Location-only LR", location_xg),
    evaluate_probabilities("Context-aware LR", context_xg),
])
pd.set_option("display.width", 200)
display(metrics.round(4))

### 5.8.1 ROC curve

How well each model separates goals from non-goals across all thresholds. The closer the curve is to the top-left corner, the better.

In [ ]:
plt.figure(figsize=(6, 5))
for label, probabilities in [("Location-only", location_xg),
                             ("Context-aware", context_xg)]:
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate (recall)")
plt.title("Logistic Regression ROC curves (held-out matches)")
plt.legend()
plt.tight_layout()
plt.show()

### 5.8.2 Precision-recall curve

Since goals are rare, the precision-recall curve is more informative than ROC on its own. The dashed line is what a random model would get (the test goal rate).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for label, probabilities in [("Location-only", location_xg),
                             ("Context-aware", context_xg)]:
    PrecisionRecallDisplay.from_predictions(y_test, probabilities, name=label, ax=ax)
ax.axhline(y_test.mean(), color="black", linestyle="--", label="Test goal rate")
ax.set_title("Logistic Regression precision–recall curves (held-out matches)")
ax.legend()
plt.tight_layout()
plt.show()

### 5.8.3 Calibration

This compares the predicted xG with how often those shots actually went in. If shots with xG around 0.20 score about 20% of the time, the model is well calibrated there. Points on the diagonal mean perfect calibration. Logistic Regression is trained by minimizing log loss, so it is usually well calibrated without extra steps.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(y_test, location_xg, n_bins=8,
                                    name="Location-only", ax=ax)
CalibrationDisplay.from_predictions(y_test, context_xg, n_bins=8,
                                    name="Context-aware", ax=ax)
ax.set_title("Logistic Regression probability calibration (held-out matches)")
plt.tight_layout()
plt.show()

### 5.8.4 Confusion matrix (threshold 0.5)

Correct and wrong predictions for the context-aware model when a shot is labelled a goal only if xG >= 0.5.

In [ ]:
context_labels = (context_xg >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_test, context_labels, display_labels=["No goal", "Goal"], cmap="Blues"
)
plt.title("Context-aware LR confusion matrix (threshold = 0.5)")
plt.tight_layout()
plt.show()

### 5.8.5 Results per class

Precision, recall and F1 for goals and non-goals separately, at threshold 0.5.

In [ ]:
for name, probabilities in [("Location-only", location_xg),
                            ("Context-aware", context_xg)]:
    print(f"\n{name} LR (threshold = 0.5)")
    labels = (probabilities >= 0.5).astype(int)
    print(classification_report(y_test, labels,
          target_names=["No goal", "Goal"], digits=3, zero_division=0))

### 5.8.6 Picking a better threshold

A threshold of 0.5 only labels a shot as a goal when the model is more than 50% sure, and that rarely happens. If we want yes/no predictions, a lower threshold gives a better balance between precision and recall.

We choose the threshold on the training data, not the test set. Each training shot gets an out-of-fold prediction from a model that did not see its match, and We pick the threshold with the best F1 on those predictions. Then We apply it to the test set. This only changes how probabilities are turned into labels; the xG values stay the same.

In [ ]:
thresholds = np.arange(0.05, 0.61, 0.01)
threshold_rows = []
chosen_thresholds = {}

for name, model, X_train_set, X_test_set, probabilities in [
    ("Location-only", location_model, X_train_location, X_test_location, location_xg),
    ("Context-aware", context_model, X_train_context, X_test_context, context_xg),
]:
    # out-of-fold predictions on the training set
    oof = cross_val_predict(model, X_train_set, y_train, cv=cv_folds,
                            method="predict_proba")[:, 1]
    f1_scores = [f1_score(y_train, oof >= t) for t in thresholds]
    best_t = float(thresholds[int(np.argmax(f1_scores))])
    chosen_thresholds[name] = best_t
    plt.plot(thresholds, f1_scores, label=f"{name} (best = {best_t:.2f})")

    labels = (probabilities >= best_t).astype(int)
    threshold_rows.append({
        "Model": f"{name} LR",
        "Threshold (from training CV)": best_t,
        "Test precision": precision_score(y_test, labels, zero_division=0),
        "Test recall": recall_score(y_test, labels),
        "Test F1": f1_score(y_test, labels),
    })

plt.xlabel("Decision threshold")
plt.ylabel("Out-of-fold F1 (training)")
plt.title("Choosing a threshold on training data")
plt.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame(threshold_rows).round(4))

In [ ]:
best_t = chosen_thresholds["Context-aware"]
ConfusionMatrixDisplay.from_predictions(
    y_test, (context_xg >= best_t).astype(int),
    display_labels=["No goal", "Goal"], cmap="Blues"
)
plt.title(f"Context-aware LR confusion matrix (threshold = {best_t:.2f})")
plt.tight_layout()
plt.show()

## 5.9 Best model and findings

The best model is chosen by cross-validation log loss, the same criterion used in tuning, and not by test results. After that We compare the two feature sets and the baseline on the test set.

In [ ]:
# Choose the best model by CV log loss (best_score_ is negative log loss).
if location_search.best_score_ >= context_search.best_score_:
    best_name, best_search = "Location-only", location_search
else:
    best_name, best_search = "Context-aware", context_search

print(f"Best LR by cross-validation log loss: {best_name}")
print("Best settings:", best_search.best_params_)
print(f"Mean CV log loss: {-best_search.best_score_:.4f}")

location_row = metrics.loc[metrics["Model"] == "Location-only LR"].iloc[0]
context_row = metrics.loc[metrics["Model"] == "Context-aware LR"].iloc[0]
baseline_row = metrics.loc[metrics["Model"] == "Training-rate baseline"].iloc[0]

print("\nTest-set results (location → context):")
for metric in ["ROC AUC", "Average precision", "Log loss", "Brier score", "Recall"]:
    print(f"{metric}: {location_row[metric]:.4f} → {context_row[metric]:.4f}")
print(f"Training-rate baseline: log loss = {baseline_row['Log loss']:.4f}, "
      f"Brier score = {baseline_row['Brier score']:.4f}")

print("\nInterpretation:")
if context_row["ROC AUC"] > location_row["ROC AUC"]:
    print("Context features improved the ranking of scoring chances on held-out matches.")
else:
    print("Context features did not improve ROC AUC on held-out matches.")
if (context_row["Log loss"] < location_row["Log loss"] and
        context_row["Brier score"] < location_row["Brier score"]):
    print("Context features also improved both probability-error measures.")
else:
    print("Context features did not improve both probability-error measures.")
print("Classification recall at 0.5 should be interpreted cautiously for rare goals.")

## 5.10 Saving the results

The test predictions are saved in the same format as `svm_predictions.csv` (shot ID, match ID, true label and the two xG columns) so both models can be compared on the same shots in `05_model_evaluation.ipynb`. We also save the metrics, the coefficients and the two trained pipelines.

In [ ]:
results_dir = project_root / "results"
models_dir = project_root / "models"
results_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

predictions = test[["id", "match_id", "goal"]].copy()
predictions["logistic_location_xg"] = location_xg
predictions["logistic_context_xg"] = context_xg
predictions.to_csv(results_dir / "logistic_predictions.csv", index=False)
metrics.to_csv(results_dir / "logistic_metrics.csv", index=False)
context_coefficients.to_csv(results_dir / "logistic_context_coefficients.csv", index=False)

joblib.dump(location_model, models_dir / "logistic_location.joblib")
joblib.dump(context_model, models_dir / "logistic_context.joblib")

print("Saved:")
print("- results/logistic_predictions.csv")
print("- results/logistic_metrics.csv")
print("- results/logistic_context_coefficients.csv")
print("- models/logistic_location.joblib")
print("- models/logistic_context.joblib")
print("Test-shot prediction rows:", len(predictions))

## 5.11 Summary

- Adding the context features improved the model on the test set in every main metric: ROC AUC went from about 0.75 to 0.80, and both log loss and Brier score went down. So how a shot is taken adds real information on top of where it is taken from.
- The final models were trained with the best settings: L2 regularization, C = 10 and no class weighting (section 5.5). L1 gave almost the same results and only dropped three weak features.
- Balanced class weights did not help. The ranking stayed the same but the probabilities became too high, which is bad for xG.
- The predicted probabilities are well calibrated and the average xG matches the real goal rate, so no extra calibration step was needed.
- Some coefficients (like corners or backheels) come from very few shots and should not be over-interpreted.



# 6. Support Vector Machine

We compare linear, RBF, and polynomial SVM kernels. Because an SVM decision score is not a probability, the selected models are calibrated on match-separated training folds before evaluation.


In [ ]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV


## 6.2 Load the dataset

The exported `modeling_dataset.csv` is intentionally **unscaled and unencoded**. It contains the engineered features, but scaling and encoding must be fitted on training data only to avoid data leakage.

`model_split.csv` contains the shared train/test assignments. We match the files by shot ID and match ID, then check that no match appears in both sets.

In [ ]:
data = pd.read_csv(project_root / "data/processed/modeling_dataset.csv")
splits = pd.read_csv(project_root / "data/processed/model_split.csv")

# Match split assignments by IDs, not by row order.
assert data["id"].is_unique and splits["id"].is_unique
shots = data.merge(splits, on=["id", "match_id"], validate="one_to_one")
assert len(shots) == len(data) == len(splits)
assert set(shots["split"]) == {"train", "test"}
assert set(shots["goal"].unique()) == {0, 1}
assert shots[CONTEXT_FEATURES].notna().all().all()

train = shots[shots["split"] == "train"].reset_index(drop=True)
test = shots[shots["split"] == "test"].reset_index(drop=True)

# Keep all shots from a match in the same partition.
assert set(train["match_id"]).isdisjoint(set(test["match_id"]))
print("Total shots:", len(shots), "| Total matches:", shots["match_id"].nunique())
print("Training:", len(train), "shots |", train["match_id"].nunique(), "matches")
print("Testing: ", len(test), "shots |", test["match_id"].nunique(), "matches")
print("Match overlap: 0")

In [ ]:
# Check class balance in both partitions.
class_counts = shots.groupby("split")["goal"].agg(
    shots="count", goals="sum", goal_rate="mean"
)
class_counts["non_goals"] = class_counts["shots"] - class_counts["goals"]
display(class_counts[["shots", "non_goals", "goals", "goal_rate"]].round(4))

## 6.3 Features and cross-validation

SVM learns a boundary between the goal and non-goal classes. We compare a model using just shot location with one using all eight features.

The preprocessing pipeline scales numeric features and encodes categorical features. Because it is inside the model pipeline, it is fitted again within each training fold.

We use **3-fold `StratifiedGroupKFold`**, grouping shots by `match_id`. Both feature sets use the same folds, and test matches are excluded from cross-validation.

In [ ]:
y_train = train["goal"]
y_test = test["goal"]

X_train_location = train[LOCATION_FEATURES]
X_test_location = test[LOCATION_FEATURES]
X_train_context = train[CONTEXT_FEATURES]
X_test_context = test[CONTEXT_FEATURES]

# Reuse these folds for both SVM experiments.
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
cv_folds = list(cv.split(X_train_location, y_train, groups=train["match_id"]))

for number, (train_idx, val_idx) in enumerate(cv_folds, 1):
    train_matches = set(train.loc[train_idx, "match_id"])
    val_matches = set(train.loc[val_idx, "match_id"])
    assert train_matches.isdisjoint(val_matches)
    print(f"Fold {number}: {len(train_idx)} training shots, {len(val_idx)} validation shots")

## 6.4 Kernel comparison and hyperparameter tuning

We compare three SVM kernels:

- **Linear:** a straight decision boundary.
- **RBF:** a flexible nonlinear boundary.
- **Polynomial:** a boundary based on polynomial relationships (`coef0=1` allows lower-degree terms).

`GridSearchCV` tests combinations of **`C`** (regularization), **`gamma`** (influence of points in nonlinear kernels), and **`degree`** (polynomial complexity).

Only about **11% of shots are goals**. We therefore test both regular and **balanced class weights**, which give the rarer goal class more importance during training. This does not change the original class distribution.

The best settings are chosen by **mean ROC AUC across three match-separated validation folds**. The test set is not used for tuning.

In [ ]:
# Compare the usual SVM against one that gives more weight to rare goals.
parameter_grid = [
    {"svm__kernel": ["linear"], "svm__C": [0.1, 1, 10],
     "svm__class_weight": [None, "balanced"]},
    {"svm__kernel": ["rbf"], "svm__C": [0.1, 1, 10],
     "svm__gamma": ["scale", 0.1], "svm__class_weight": [None, "balanced"]},
    {"svm__kernel": ["poly"], "svm__C": [0.1],
     "svm__gamma": ["scale"], "svm__degree": [2, 3],
     "svm__coef0": [1],
     "svm__class_weight": [None, "balanced"]},
]

print("Grid candidates per feature set: 22 (each tested across 3 folds)")

### 6.4.1 Location-only SVM

Train an SVM using only `distance` and `angle`. Grid search tests the kernel settings on the three training folds and reports the best result.

In [ ]:
location_pipeline = Pipeline([
    ("preprocessing", make_location_preprocessor()),
    ("svm", SVC()),
])

location_search = GridSearchCV(
    location_pipeline, parameter_grid, scoring="roc_auc",
    cv=cv_folds, n_jobs=1
)
start = time.perf_counter()
location_search.fit(X_train_location, y_train)
location_seconds = time.perf_counter() - start

print("Location-only best parameters:", location_search.best_params_)
print("Location-only best CV ROC AUC:", round(location_search.best_score_, 4))
print(f"Grid search time: {location_seconds:.1f} seconds")

In [ ]:
location_cv = pd.DataFrame(location_search.cv_results_)
location_settings = location_cv[[
    "param_svm__kernel", "param_svm__C", "param_svm__gamma",
    "param_svm__degree", "param_svm__coef0", "param_svm__class_weight",
    "mean_test_score", "std_test_score",
]].copy()
location_settings.columns = [
    "Kernel", "C", "Gamma", "Degree", "Coef0", "Class weight", "Mean CV AUC", "CV std. dev."
]
display(location_settings.fillna("-").sort_values("Mean CV AUC", ascending=False).round(4))

### 6.4.2 Context-aware SVM

Repeat the same experiment using all eight features and the **same validation folds**, so the comparison is fair.

In [ ]:
context_pipeline = Pipeline([
    ("preprocessing", make_context_preprocessor()),
    ("svm", SVC()),
])

context_search = GridSearchCV(
    context_pipeline, parameter_grid, scoring="roc_auc",
    cv=cv_folds, n_jobs=1
)
start = time.perf_counter()
context_search.fit(X_train_context, y_train)
context_seconds = time.perf_counter() - start

print("Context-aware best parameters:", context_search.best_params_)
print("Context-aware best CV ROC AUC:", round(context_search.best_score_, 4))
print(f"Grid search time: {context_seconds:.1f} seconds")

In [ ]:
context_cv = pd.DataFrame(context_search.cv_results_)
context_settings = context_cv[[
    "param_svm__kernel", "param_svm__C", "param_svm__gamma",
    "param_svm__degree", "param_svm__coef0", "param_svm__class_weight",
    "mean_test_score", "std_test_score",
]].copy()
context_settings.columns = [
    "Kernel", "C", "Gamma", "Degree", "Coef0", "Class weight", "Mean CV AUC", "CV std. dev."
]
display(context_settings.fillna("-").sort_values("Mean CV AUC", ascending=False).round(4))

### 6.4.3 Compare kernels and class weights

The table and chart compare the strongest validation ROC AUC for each kernel. The second table shows whether balancing the classes helped during tuning. These results use training folds only.

In [ ]:
kernel_results = []
for feature_set, search in [("Location-only", location_search),
                            ("Context-aware", context_search)]:
    cv_results = pd.DataFrame(search.cv_results_)
    for kernel in ["linear", "rbf", "poly"]:
        selected = cv_results[cv_results["param_svm__kernel"] == kernel]
        kernel_results.append({
            "Feature set": feature_set,
            "Kernel": kernel,
            "Best CV ROC AUC": selected["mean_test_score"].max(),
        })

kernel_table = pd.DataFrame(kernel_results)
display(kernel_table.round(4))

kernel_plot = kernel_table.pivot(index="Kernel", columns="Feature set", values="Best CV ROC AUC")
kernel_plot.plot(kind="bar", figsize=(7, 4), rot=0)
plt.ylabel("Best mean CV ROC AUC")
plt.title("SVM kernel comparison (training CV)")
plt.tight_layout()
plt.show()

In [ ]:
weight_rows = []
for name, search in [("Location-only", location_search), ("Context-aware", context_search)]:
    results = pd.DataFrame(search.cv_results_)
    results["Class weight"] = results["param_svm__class_weight"].fillna("None")
    for weight in ["None", "balanced"]:
        selected = results[results["Class weight"] == weight]
        weight_rows.append({
            "Feature set": name,
            "Class weight": weight,
            "Best mean CV ROC AUC": selected["mean_test_score"].max(),
        })

display(pd.DataFrame(weight_rows).round(4))

## 6.5 Estimate xG probabilities

A regular SVM produces a decision score rather than a scoring probability. We use **sigmoid calibration** to turn those scores into xG values between 0 and 1.

`CalibratedClassifierCV` uses match-separated training folds for calibration. We use this instead of `SVC(probability=True)` so that the calibration folds also respect match boundaries. The test set remains untouched.

In [ ]:
location_model = CalibratedClassifierCV(
    estimator=location_search.best_estimator_, method="sigmoid",
    cv=cv_folds, ensemble=False
)
context_model = CalibratedClassifierCV(
    estimator=context_search.best_estimator_, method="sigmoid",
    cv=cv_folds, ensemble=False
)

location_model.fit(X_train_location, y_train)
context_model.fit(X_train_context, y_train)
print("Both selected SVMs fitted with match-separated sigmoid calibration.")

In [ ]:
# Column 1 gives P(goal = 1), which is our xG estimate.
location_xg = location_model.predict_proba(X_test_location)[:, 1]
context_xg = context_model.predict_proba(X_test_context)[:, 1]

example_predictions = test[["id", "goal"]].head(5).copy()
example_predictions["Location xG"] = location_xg[:5]
example_predictions["Context xG"] = context_xg[:5]
display(example_predictions.round(4))

### 6.5.1 Check predicted probabilities

The model should assign different xG values to shots with different scoring chances. The summary below helps check whether predictions vary meaningfully instead of staying near the average goal rate.

In [ ]:
probability_summary = pd.DataFrame({
    "Feature set": ["Location-only", "Context-aware"],
    "Mean xG": [location_xg.mean(), context_xg.mean()],
    "Std. dev.": [location_xg.std(), context_xg.std()],
    "Min xG": [location_xg.min(), context_xg.min()],
    "Median xG": [np.median(location_xg), np.median(context_xg)],
    "Max xG": [location_xg.max(), context_xg.max()],
})
display(probability_summary.round(4))

## 6.6 Preliminary evaluation

Most shots are non-goals, so **accuracy alone is misleading**. We report precision, recall and F1 for classification, ROC AUC and average precision for ranking, and **log loss and Brier score** for probability quality.

We also compare against a simple baseline that gives every shot the training set's average goal probability. Classification metrics use a threshold of **0.5**; low recall at this threshold does not necessarily mean the xG probabilities are unhelpful.

In [ ]:
# The baseline predicts the same training goal rate for every shot.
training_goal_rate = y_train.mean()
baseline_xg = np.full(len(test), training_goal_rate)

def evaluate_probabilities(name, probabilities):
    labels = (probabilities >= 0.5).astype(int)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, labels),
        "Precision": precision_score(y_test, labels, zero_division=0),
        "Recall": recall_score(y_test, labels, zero_division=0),
        "F1": f1_score(y_test, labels, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, probabilities),
        "Average precision": average_precision_score(y_test, probabilities),
        "Log loss": log_loss(y_test, probabilities),
        "Brier score": brier_score_loss(y_test, probabilities),
    }

metrics = pd.DataFrame([
    evaluate_probabilities("Training-rate baseline", baseline_xg),
    evaluate_probabilities("Location-only SVM", location_xg),
    evaluate_probabilities("Context-aware SVM", context_xg),
])
display(metrics.round(4))

### 6.6.1 ROC curve

The ROC curve shows how well each model separates goals from non-goals across different thresholds. A higher ROC AUC indicates better ranking.

In [ ]:
plt.figure(figsize=(6, 5))
for label, probabilities in [("Location-only", location_xg),
                             ("Context-aware", context_xg)]:
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate (recall)")
plt.title("SVM ROC curves (held-out matches)")
plt.legend()
plt.tight_layout()
plt.show()

### 6.6.2 Precision–recall curve

Goals are uncommon, so a precision–recall curve is useful alongside ROC AUC. Higher curves indicate better precision at a given recall. The dashed line shows the goal rate in the test set.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for label, probabilities in [("Location-only", location_xg),
                             ("Context-aware", context_xg)]:
    PrecisionRecallDisplay.from_predictions(y_test, probabilities, name=label, ax=ax)
ax.axhline(y_test.mean(), color="black", linestyle="--", label="Test goal rate")
ax.set_title("SVM precision–recall curves (held-out matches)")
ax.legend()
plt.tight_layout()
plt.show()

### 6.6.3 Calibration curve

This plot compares predicted xG with actual goal rates. If shots predicted at about 0.20 xG score around 20% of the time, the probabilities are well calibrated in that range. The diagonal represents ideal calibration.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(y_test, location_xg, n_bins=8,
                                    name="Location-only", ax=ax)
CalibrationDisplay.from_predictions(y_test, context_xg, n_bins=8,
                                    name="Context-aware", ax=ax)
ax.set_title("SVM probability calibration (held-out matches)")
plt.tight_layout()
plt.show()

### 6.6.4 Confusion matrix

The confusion matrix shows correct and incorrect classifications for the context-aware model at a **0.5 threshold**. Because goals are rare, many shots may be predicted as non-goals even when their xG probabilities are useful.

In [ ]:
context_labels = (context_xg >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_test, context_labels, display_labels=["No goal", "Goal"], cmap="Blues"
)
plt.title("Context-aware SVM confusion matrix (threshold = 0.5)")
plt.tight_layout()
plt.show()

### 6.6.5 Per-class classification results

This report shows precision, recall and F1 separately for **goals** and **non-goals** using the usual 0.5 threshold. A low goal recall does not by itself mean xG estimates are poor.

In [ ]:
for name, probabilities in [("Location-only", location_xg),
                            ("Context-aware", context_xg)]:
    print(f"\n{name} SVM (threshold = 0.5)")
    labels = (probabilities >= 0.5).astype(int)
    print(classification_report(y_test, labels,
          target_names=["No goal", "Goal"], digits=3, zero_division=0))

## 6.7 Findings and best SVM

We select the **best SVM by cross-validation ROC AUC** (the criterion used during tuning). Then we compare its held-out results with the other feature set and the training-rate baseline.

ROC AUC and average precision measure how well shots are ranked. Log loss and Brier score measure the accuracy of **xG probabilities**. These measures may favor different models.

In [ ]:
# Select the overall best SVM using validation performance, not the test set.
if location_search.best_score_ >= context_search.best_score_:
    best_name, best_search = "Location-only", location_search
else:
    best_name, best_search = "Context-aware", context_search

print(f"Best SVM by cross-validation ROC AUC: {best_name}")
print("Best settings:", best_search.best_params_)
print(f"Mean CV ROC AUC: {best_search.best_score_:.4f}")

location_row = metrics.loc[metrics["Model"] == "Location-only SVM"].iloc[0]
context_row = metrics.loc[metrics["Model"] == "Context-aware SVM"].iloc[0]
baseline_row = metrics.loc[metrics["Model"] == "Training-rate baseline"].iloc[0]

print("\nTest-set results (location → context):")
for metric in ["ROC AUC", "Average precision", "Log loss", "Brier score", "Recall"]:
    print(f"{metric}: {location_row[metric]:.4f} → {context_row[metric]:.4f}")
print(f"Training-rate baseline: log loss = {baseline_row['Log loss']:.4f}, "
      f"Brier score = {baseline_row['Brier score']:.4f}")

print("\nInterpretation:")
if context_row["ROC AUC"] > location_row["ROC AUC"]:
    print("Context features improved the ranking of scoring chances on held-out matches.")
else:
    print("Context features did not improve ROC AUC on held-out matches.")
if (context_row["Log loss"] < location_row["Log loss"] and
        context_row["Brier score"] < location_row["Brier score"]):
    print("Context features also improved both probability-error measures.")
else:
    print("Context features did not improve both probability-error measures.")
print("Classification recall at 0.5 should be interpreted cautiously for rare goals.")

## 6.8 Save results for model comparison

Save the test-shot IDs, true labels and both xG predictions in `svm_predictions.csv`. We also save the metrics and fitted models, so the SVM results can be compared with Logistic Regression on the **same test shots**.

In [ ]:
results_dir = project_root / "results"
models_dir = project_root / "models"
results_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

predictions = test[["id", "match_id", "goal"]].copy()
predictions["svm_location_xg"] = location_xg
predictions["svm_context_xg"] = context_xg
predictions.to_csv(results_dir / "svm_predictions.csv", index=False)
metrics.to_csv(results_dir / "svm_metrics.csv", index=False)

joblib.dump(location_model, models_dir / "svm_location.joblib")
joblib.dump(context_model, models_dir / "svm_context.joblib")

print("Saved:")
print("- results/svm_predictions.csv")
print("- results/svm_metrics.csv")
print("- models/svm_location.joblib")
print("- models/svm_context.joblib")
print("Test-shot prediction rows:", len(predictions))

## 6.9 SVM Summary

This section trains SVMs with linear, RBF and polynomial kernels using both location-only and context-aware features. It compares class weights, tunes parameters with match-separated cross-validation, and calibrates SVM scores into xG probabilities.

When executed, the tables and figures show the results, including the best SVM selected by **validation ROC AUC** in Section 6.7. The exported predictions can be compared with Logistic Regression on the same test shots.



# 7. Results and Remaining Comparison

The Logistic Regression and SVM sections evaluate the same held-out matches and save predictions with shot identifiers. A consolidated comparison across all four configurations has not been added yet, so this notebook does not select an overall winner across both algorithms. That comparison should join the saved predictions by shot ID and use one clearly stated selection rule.


# 8. Current Findings

Within each algorithm, the context-aware feature set performs better than distance and angle alone on the reported held-out probability and ranking measures. The final cross-algorithm conclusion remains open until the four-model comparison is completed.
